In [1]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
import torch
from huggingface_hub import login
import os
from datasets import load_from_disk

In [2]:
# Set path for tokenized dataset
tokenized_path = "tokenized_tiny_codes_qwen2.5_0.5b"
os.environ["HF_DATASETS_CACHE"] = "E:"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
# Load dataset
login()
print("Loading tiny-codes dataset...")
ds = load_dataset("nampdn-ai/tiny-codes", split="train")

Loading tiny-codes dataset...


In [4]:
# Load model and tokenizer
model_name = "Qwen/Qwen2-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto"
)
model.gradient_checkpointing_enable()

`torch_dtype` is deprecated! Use `dtype` instead!
C:\Users\antho\miniforge3\envs\LLM\Lib\site-packages\accelerate\utils\modeling.py:821: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  _ = torch.tensor([0], device=i)


In [5]:
# Check if tokenized dataset exists
if os.path.exists(tokenized_path):
    print(f"Loading tokenized dataset from {tokenized_path}...")
    tokenized_ds = load_from_disk(tokenized_path)
else:
    print("Loading and tokenizing dataset...")
    ds = load_dataset("nampdn-ai/tiny-codes", split="train")

    def preprocess(batch):
        prompts = batch["prompt"]
        responses = batch["response"]
        results = { "input_ids": [], "attention_mask": [], "labels": [] }
        for prompt, response in zip(prompts, responses):
            messages = [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": response},
            ]
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            tokenized = tokenizer(
                text,
                max_length=2048,
                truncation=True,
                padding="max_length"
            )
            results["input_ids"].append(tokenized["input_ids"])
            results["attention_mask"].append(tokenized["attention_mask"])
            results["labels"].append(tokenized["input_ids"].copy())
        return results

    tokenized_ds = ds.map(
        preprocess,
        batched=True,
        batch_size=512,
        remove_columns=ds.column_names,
        desc="Tokenizing dataset"
    )
    tokenized_ds.save_to_disk(tokenized_path)

Loading tokenized dataset from tokenized_tiny_codes_qwen2.5_0.5b...


Loading dataset from disk:   0%|          | 0/87 [00:00<?, ?it/s]

In [6]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    learning_rate=.01,
    logging_steps=100,
    save_steps=500,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to=[],
)

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
    tokenizer=tokenizer,
)

# Train
trainer.train()

C:\Users\antho\AppData\Local\Temp\ipykernel_13448\3194541162.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
